In [1]:
%matplotlib qt
import matplotlib.pyplot as plt
import mne
# from neuronol_signalprocessing import perform_EMD, calculate_hilbert_spectrum
import numpy as np
import os
import pandas as pd
# from PyEMD import EMD

# Given

In [2]:
data_dir = os.path.expanduser('~/data/EEGAcamp/')
sfreq = 1000
fname_raw = "001_robin_11132024.csv"
# fname_raw = "001_Anonymous 02-10-24 14h18m.csv"
# fname_raw = "001_Anonymous 18-09-24 13h52m.csv"
# fname_raw = "001_Amira Ahamed.csv"

In [3]:
# Import raw data in CSV file and convert to a formatted dataframe
#   Read data from Excel and drop general info at top
# fpath_raw = os.path.join(data_dir, "RestingState", fname_raw)
fpath_raw = os.path.join(data_dir, "iMotions", fname_raw)
df_raw = pd.read_csv(fpath_raw, low_memory=False)
inx_first_timepoint = df_raw.index[df_raw.iloc[:, 0] == '1'].to_list()[0]
inx_header = inx_first_timepoint - 1
df_raw.drop(df_raw.head(inx_header).index, inplace=True)

#   Change header
header = df_raw.iloc[0].values
df_raw = df_raw[1:]
df_raw.columns = header
df_raw

,Row,Timestamp,EventSource,SlideEvent,StimType,Duration,CollectionPhase,SourceStimuliName,EventSource,SampleNumber,...,ET_PupilRight,ET_TimeSignal,ET_DistanceLeft,ET_DistanceRight,ET_CameraLeftX,ET_CameraLeftY,ET_CameraRightX,ET_CameraRightY,ET_ValidityLeft,ET_ValidityRight
28,1,37.1898,1,StartSlide,TestImage,3000000,StimuliDisplay,Screen recording,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29,2,38.14855,NaN,NaN,NaN,NaN,NaN,Screen recording,1,285950,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
30,3,39.14855,NaN,NaN,NaN,NaN,NaN,Screen recording,1,285951,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
31,4,40.14855,NaN,NaN,NaN,NaN,NaN,Screen recording,1,285952,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
32,5,40.477,NaN,NaN,NaN,NaN,NaN,Screen recording,NaN,NaN,...,3.77203369140625,43.333000000566244,615.10302734375,614.511962890625,0.55884528160095215,0.44084787368774414,0.37295794486999512,0.44754943251609802,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373195,373168,287045.6367,NaN,NaN,NaN,NaN,NaN,Screen recording,NaN,NaN,...,3.345611572265625,287046.12700000033,621.7755126953125,618.18011474609375,0.57473742961883545,0.43325275182723999,0.39073535799980164,0.43544071912765503,0,0
373196,373169,287046.14855,NaN,NaN,NaN,NaN,NaN,Screen recording,1,572958,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
373197,373170,287047.14855,NaN,NaN,NaN,NaN,NaN,Screen recording,1,572959,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
373198,373171,287047.9541,1,EndMedia,TestImage,3000000,StimuliDisplay,Screen recording,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# #   Extract LSL triggers (note: Timestamp are in msecs and LSL Timestamp are in secs)
# df_triggers = df_raw.drop(df_raw.index[df_raw['LSL Timestamp'].isna()])
# df_triggers.dropna(axis=1, how='all', inplace=True)  # drop columns that don't have any data left
# df_triggers.drop(columns=['Row', 'Combined Event Source', 'SourceStimuliName'], inplace=True)
# df_triggers['Timestamp'] = df_triggers['Timestamp'].astype('float')
# df_triggers.reset_index(drop=True, inplace=True)
# df_triggers

In [4]:
#   Extract EEG data
df_eeg = df_raw.drop(df_raw.index[df_raw['Fp1'].isna()])    # drop rows without EEG data
df_eeg.dropna(axis=1, how='all', inplace=True)  # drop columns that don't have any data left
df_eeg.drop(columns=['Row', 'SourceStimuliName', 'SampleNumber'],
            inplace=True)
aux_ch_names = ['Aux%d' % w for w in range(1, 9)] + ['Channel %d' % w for w in range(33, 41)]
for col_name in ['Combined Event Source', 'EventSource'] + aux_ch_names:
    try:
        df_eeg.drop(columns=[col_name], inplace=True)
    except:
        pass
df_eeg = df_eeg.astype('float')
df_eeg.reset_index(drop=True, inplace=True)
df_eeg

,Timestamp,Fp1,Fz,F3,F7,FT9,FC5,FC1,C3,T7,...,CP2,Cz,C4,T8,FT10,FC6,FC2,F4,F8,Fp2
0,38.14855,-3692.480375,4033.495992,10432.665752,5358.007677,11229.345419,13898.974258,6223.437343,12757.372725,7065.624822,...,10088.183339,7300.976378,7165.624819,-498.193347,6386.767417,5172.363151,3400.195227,5811.865088,7427.538875,-6974.755683
1,39.14855,-3621.533112,4107.665912,10506.396219,5431.005722,11307.861042,13973.534803,6298.242028,12832.128582,7149.365054,...,10163.720446,7377.197079,7237.548645,-430.224599,6463.232259,5240.917836,3475.732334,5884.374851,7501.318170,-6900.292794
2,40.14855,-3518.212802,4208.300675,10610.644263,5533.202985,11411.767290,14077.099254,6400.683432,12936.523111,7260.351379,...,10263.769272,7478.759577,7335.790830,-340.185538,6561.669756,5336.962756,3575.341707,5983.398286,7601.659964,-6800.439281
3,41.14855,-3433.007726,4293.505751,10697.558324,5620.410014,11499.706741,14164.941048,6485.497883,13024.413733,7348.388486,...,10351.171614,7564.501762,7422.167781,-246.874994,6652.050613,5416.454941,3660.644439,6065.380706,7688.622853,-6718.066237
4,42.14855,-3372.119055,4356.152234,10761.279025,5684.130716,11567.236036,14226.659797,6549.658038,13086.816076,7408.642391,...,10412.304424,7627.392385,7483.642389,-170.605464,6718.554518,5477.245955,3722.412015,6129.394376,7746.972461,-6659.228347
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
287005,287043.14855,-3173.681560,3728.662015,11151.659875,5237.060415,11964.550479,14889.257436,6749.462720,13333.446929,6415.722494,...,10546.581765,8403.368928,7258.593567,1112.402316,6424.609213,5917.236179,3848.339747,5846.337743,7802.197069,-6581.249834
287006,287044.14855,-3154.199139,3745.458890,11170.409874,5257.519398,11987.597353,14910.644155,6767.431470,13351.806303,6424.169760,...,10563.085671,8423.046662,7276.513488,1126.269503,6442.822103,5932.910006,3864.745996,5863.915867,7818.993943,-6561.523272
287007,287045.14855,-3239.697184,3660.058501,11084.667689,5170.947135,11903.027043,14826.073844,6679.199050,13266.943024,6333.349449,...,10473.583720,8336.767368,7188.964662,1032.470677,6351.171715,5841.357274,3779.980373,5778.076026,7728.857227,-6645.312332
287008,287046.14855,-3368.163977,3528.857333,10951.708708,5034.667842,11758.398140,14690.185176,6546.240069,13132.519199,6199.902187,...,10341.113020,8202.245887,7056.884587,893.896462,6210.204921,5709.130715,3647.851470,5648.730326,7590.283011,-6778.222485


In [5]:
# Convert dataframe to numpy array in SI units
eeg_data = df_eeg.iloc[:, 1:].values.T / 1e6    # in Volts

# Create MNE info object for the data
ch_names = df_eeg.columns.tolist()[1:]
ch_types = ['eeg'] * len(ch_names)
info = mne.create_info(ch_names, sfreq, ch_types=ch_types)

# Create MNE raw object for the data
raw = mne.io.RawArray(eeg_data, info)
# raw.crop(0, 12)

# Add GND channel
mne.add_reference_channels(raw, ref_channels=['GND'], copy=False)
raw

Creating RawArray with float64 data, n_channels=32, n_times=287010
    Range : 0 ... 287009 =      0.000 ...   287.009 secs
Ready.
Location for this channel is unknown; consider calling set_montage() after adding new reference channels if needed. Applying a montage will only set locations of channels that exist at the time it is applied.


Measurement date,Unknown
Experimenter,Unknown
Participant,Unknown
Digitized points,Not available
Good channels,33 EEG
Bad channels,None
EOG channels,Not available
ECG channels,Not available
Sampling frequency,1000.00 Hz
Highpass,0.00 Hz
Lowpass,500.00 Hz


In [7]:
# Export raw data
fpath_exportedraw = os.path.join(data_dir, "iMotions", "sub-01_raw_eeg.fif")
raw.save(fpath_exportedraw, overwrite=True)

Overwriting existing file.
Writing /Users/chholakp2/data/EEGAcamp/iMotions/sub-01_raw_eeg.fif
Closing /Users/chholakp2/data/EEGAcamp/iMotions/sub-01_raw_eeg.fif
[done]


In [ ]:
# Plot frequency power spectra
# raw.compute_psd(fmax=100).plot()

# Plot time series
# raw.plot(scalings='auto')

# Filter raw data and plot time series
raw_filtered = raw.copy().filter(l_freq=None, h_freq=50)
raw_filtered.plot(scalings='auto', duration=5)

## HHT Demo

In [ ]:
# Perform EMD-HHT on a segment of Channel O1
n_seg = 10000
x = df_eeg['O1'].values[-n_seg:]
t = df_eeg['Timestamp'].values[-n_seg:] / sfreq

# Perform EMD-HHT
imfs = perform_EMD(x, t=t, plot_emd=False)
print('Found a total of %02d IMFs' % len(imfs))

In [7]:
# Visualize EMD output
# imfs_subset = imfs[:10]
imfs_subset = imfs
# fname_fig = 'Demo-EMD.pdf'
# fpath_fig = os.path.join(dir_results, fname_fig)
plt.figure(figsize=(12, 12))
for i in range(len(imfs_subset)-1):
    plt.subplot(len(imfs_subset), 1, i+1)
    plt.plot(t, x, color='0.8')
    plt.plot(t, imfs_subset[i], 'k')
    plt.xticks([])
    plt.xlim([np.min(t), np.max(t)])
    plt.ylabel('IMF ' + str(i + 1))
plt.subplot(len(imfs_subset), 1, i+2)
plt.plot(t, x, color='0.8')
plt.plot(t, imfs_subset[-1], 'k')
plt.xlim([np.min(t), np.max(t)])
# plt.ylabel('Residual')
plt.xlabel('Time (s)')
plt.tight_layout()
# plt.savefig(fpath_fig, bbox_inches='tight')
# plt.savefig(fpath_fig[:-4] + '.png', bbox_inches='tight') # also save as png
plt.show()

In [14]:
C = imfs[2:-1]
hht, t_hht, f_hht, f_hht_marginal, marginal_spec = calculate_hilbert_spectrum(C, t, sfreq,
                                    compute_power_spec=True, smoothing_downsample_freq=True,
                                    smoothing_gauss_filt=True, plot_inst_freq=True)

In [15]:
# Make custom demo plot
# fpath_fig = os.path.join(dir_results, 'Demo-HHT_Power.pdf')
fig, (ax1, ax2) = plt.subplots(1, 2, gridspec_kw={'width_ratios': [2.5, 1]}, figsize=(18, 8))

plt.suptitle('Demo: HHT')

im = ax1.pcolormesh(t_hht, f_hht, hht, cmap='hot')
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Frequency (Hz)')
ax1.set_title('Hilbert Power Spectrum')
fig.colorbar(im, ax=ax1)

ax2.plot(f_hht_marginal, marginal_spec)
ax2.set_xlabel('Frequency (Hz)')
ax2.set_ylabel('Power')
ax2.set_xlim([0, 50])
ax2.set_title('Marginal Hilbert Power Spectrum')

fig.tight_layout()
# plt.savefig(fpath_fig, bbox_inches='tight')
# plt.savefig(fpath_fig[:-4] + '.png', bbox_inches='tight') # also save as png
plt.show()